**texte en gras**# 📊 EDA — ST2 : Segmentation RFM, CLV & Fidélisation
**Projet 2 — Segmentation RFM, CLV & Optimisation E-Commerce**

Ce notebook analyse la base client à partir des tables Gold dbt :
- `mart_rfm`     — scores R, F, M + segment nommé par client
- `mart_clv`     — Customer Lifetime Value simplifiée par client
- `mart_cohorts` — rétention mensuelle par cohorte d'acquisition
- `stg_shipping` — satisfaction client (corrélation avec RFM)

**Questions métier couvertes (CDC §4.2) :**
1. Combien de segments RFM distincts et quelle taille relative ?
2. Quel segment représente le plus grand risque de churn (At Risk) ?
3. Quelle est la CLV moyenne par segment et l'écart avec le CAC estimé ?
4. Comment évolue la rétention mensuelle par cohorte ?
5. Y a-t-il une corrélation entre satisfaction client et segment RFM ?
6. Les clients fidèles sont-ils moins sensibles aux promotions ?

## 0. Setup & Connexion à DuckDB Gold

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Imports ───────────────────────────────────────────────
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Connexion à la base Gold (créée par dbt run) ──────────
GOLD_DB = '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/gold.duckdb'
con = duckdb.connect(GOLD_DB, read_only=True)

# ── Vérifier les tables disponibles ───────────────────────
tables = con.sql('SHOW TABLES').fetchdf()
print('Tables disponibles dans Gold:')
print(tables)

Tables disponibles dans Gold:
                 name
0        dim_customer
1            dim_date
2         dim_payment
3        fact_returns
4   fact_transactions
5            mart_clv
6        mart_cohorts
7         mart_funnel
8            mart_rfm
9         stg_returns
10       stg_shipping
11   stg_transactions


In [ ]:
# ── Charger les tables utiles pour ST2 ────────────────────
df_rfm      = con.sql('SELECT * FROM mart_rfm').fetchdf()
df_clv      = con.sql('SELECT * FROM mart_clv').fetchdf()
df_cohorts  = con.sql('SELECT * FROM mart_cohorts').fetchdf()
df_shipping = con.sql('SELECT * FROM stg_shipping').fetchdf()

print(f'mart_rfm      : {len(df_rfm):,} clients segmentés')
print(f'mart_clv      : {len(df_clv):,} clients avec CLV')
print(f'mart_cohorts  : {len(df_cohorts):,} lignes cohorte x période')
print(f'stg_shipping  : {len(df_shipping):,} commandes shipping')
print()
print('Colonnes mart_rfm :')
print(df_rfm.dtypes)
print()
print('Colonnes mart_clv :')
print(df_clv.dtypes)
print('Colonnes mart_cohorts :')
print(df_cohorts.dtypes)
print('Colonnes stg_shipping :')
print(df_shipping.dtypes)

mart_rfm      : 4,253 clients segmentés
mart_clv      : 4,253 clients avec CLV
mart_cohorts  : 91 lignes cohorte x période
stg_shipping  : 10,999 commandes shipping

Colonnes mart_rfm :
customer_id      object
country          object
recency_days      int64
frequency         int64
monetary        float64
r_score           int64
f_score           int64
m_score           int64
rfm_score       float64
rfm_segment      object
dtype: object

Colonnes mart_clv :
customer_id                   object
avg_basket                   float64
total_orders                   int64
total_revenue                float64
relation_months                int64
first_order_date      datetime64[ns]
last_order_date       datetime64[ns]
annual_frequency             float64
relation_years               float64
clv_simple                   float64
retention_cost_est           float64
clv_net                      float64
rfm_segment                   object
rfm_score                    float64
r_score              

## 1. Question métier 1 — Combien de segments RFM et quelle taille relative ?

In [ ]:
# ── Distribution des segments RFM ─────────────────────────
seg_dist = df_rfm.groupby('rfm_segment').agg(
    nb_clients  = ('customer_id', 'count'),
    ca_total    = ('monetary',    'sum'),
    ca_moyen    = ('monetary',    'mean'),
    freq_moy    = ('frequency',   'mean'),
    recence_moy = ('recency_days','mean')
).reset_index()#transforme le résultat groupby en tableau classique

total_clients = seg_dist['nb_clients'].sum()
total_ca      = seg_dist['ca_total'].sum()

seg_dist['pct_clients'] = (seg_dist['nb_clients'] / total_clients * 100).round(1)#un nombre à 1 chiffre après la virgule
seg_dist['pct_ca']      = (seg_dist['ca_total']   / total_ca      * 100).round(1)
seg_dist = seg_dist.sort_values('ca_total', ascending=False)#classe les segments du plus rentable au moins rentable le ca plus grand

print('=' * 70)
print('  DISTRIBUTION DES SEGMENTS RFM')
print('=' * 70)
print(seg_dist[['rfm_segment','nb_clients','pct_clients','ca_total','pct_ca',
                'ca_moyen','freq_moy','recence_moy']].to_string(index=False))
#affiche le DataFrame comme un tableau lisible : et surtout : sans la colonne d’index (0, 1, 2, 3…)
print('=' * 70)
print(f'  Total clients   : {total_clients:,}')
print(f'  Total CA        : {total_ca:,.2f} GBP')#GBP = British Pound Sterling C’est la monnaie officielle du Royaume-Uni.
print('=' * 70)

  DISTRIBUTION DES SEGMENTS RFM
        rfm_segment  nb_clients  pct_clients   ca_total  pct_ca    ca_moyen  freq_moy  recence_moy
          Champions         986         23.2 3462252.69    61.8 3511.412465 10.110548    13.469574
    Loyal Customers        1187         27.9 1471607.52    26.3 1239.770447  3.895535    66.849200
        Hibernating         670         15.8  335164.01     6.0  500.244791  1.261194   218.598507
Potential Loyalists         527         12.4  167239.29     3.0  317.342106  1.000000    38.017078
               Lost         766         18.0  137273.00     2.4  179.207572  1.514360   159.449086
            At Risk         117          2.8   32271.23     0.6  275.822479  2.709402   133.709402
  Total clients   : 4,253
  Total CA        : 5,605,807.74 GBP


In [ ]:
# ── Palette couleur par segment (cohérente dans tout le notebook) ──
SEGMENT_COLORS = {
    'Champions'          : '#2ECC71',
    'Loyal Customers'    : '#3498DB',
    'Potential Loyalists': '#9B59B6',
    'New Customers'      : '#1ABC9C',
    'At Risk'            : '#E74C3C',
    'Hibernating'        : '#F39C12',
    'Lost'               : '#95A5A6'
}#C’est un dictionnaire Python qui associe :un segment RFM → une couleur

# ── Visualisation : Treemap segments (volume + CA) ────────:Un treemap est un type de graphique utilisé pour visualiser des données hiérarchiques sous forme de rectangles
fig_treemap = px.treemap(
    seg_dist,
    path   = ['rfm_segment'],#je veux découper par segment RFM
    values = 'nb_clients',#la taille du rectangle = nombre de clients plus il y a de clients → plus le bloc est grand
    color  = 'pct_ca',
    color_continuous_scale = 'RdYlGn',#palette automatique : rouge = mauvais jaune = moyen vert = bon
    hover_data = ['pct_clients', 'ca_total', 'pct_ca'],#quand tu passes la souris :tu vois :% clients CA total % CA
    title  = 'Q1 — Treemap segments RFM : taille = nb clients | couleur = % CA'
)
fig_treemap.update_layout(height=480)#fixe la hauteur du graphique
fig_treemap.show()

In [ ]:
# ── Visualisation : Donut double — Clients vs CA ──────────
fig_donut = make_subplots(
    rows=1, cols=2,#ça veut dire qu’on crée une grille de 2 emplacements côte à côte
    specs=[[{'type':'pie'}, {'type':'pie'}]],
    subplot_titles=['Répartition clients (%)', 'Répartition CA (%)']
)

colors = [SEGMENT_COLORS.get(s, '#BDC3C7') for s in seg_dist['rfm_segment']]
#get(s, '#BDC3C7')signifie :si le segment existe → couleur définiesinon → gris par défaut*#
fig_donut.add_trace(go.Pie(
    labels=seg_dist['rfm_segment'],
    values=seg_dist['nb_clients'],
    hole=0.45,#trou au centre
    marker_colors=colors,
    name='Clients'
), row=1, col=1)

fig_donut.add_trace(go.Pie(
    labels=seg_dist['rfm_segment'],
    values=seg_dist['ca_total'],
    hole=0.45,
    marker_colors=colors,
    name='CA'
), row=1, col=2)

fig_donut.update_layout(
    title='Q1 — Structure de la base client : volume vs valeur',
    height=420
)
fig_donut.show()

## 2. Question métier 2 — Quel segment représente le plus grand risque de churn ?

In [ ]:
# ── Segment At Risk : CA en danger dans les 90 prochains jours ──
at_risk = df_rfm[df_rfm['rfm_segment'] == 'At Risk'].copy()
hibern  = df_rfm[df_rfm['rfm_segment'] == 'Hibernating'].copy()

ca_at_risk  = at_risk['monetary'].sum()
ca_hibern   = hibern['monetary'].sum()
nb_at_risk  = len(at_risk)
nb_hibern   = len(hibern)

print('=' * 60)
print('  ANALYSE RISQUE CHURN — 90 PROCHAINS JOURS')
print('=' * 60)
print(f'  Segment At Risk')
print(f'    Nb clients             : {nb_at_risk:,}')
print(f'    CA historique en danger: {ca_at_risk:,.2f} GBP')
print(f'    % du CA total          : {ca_at_risk/total_ca*100:.1f}%')
print(f'    Récence moyenne        : {at_risk["recency_days"].mean():.0f} jours')
print(f'    Fréquence moyenne      : {at_risk["frequency"].mean():.1f} commandes')
print()
print(f'  Segment Hibernating')
print(f'    Nb clients             : {nb_hibern:,}')
print(f'    CA historique          : {ca_hibern:,.2f} GBP')
print(f'    % du CA total          : {ca_hibern/total_ca*100:.1f}%')
print(f'    Récence moyenne        : {hibern["recency_days"].mean():.0f} jours')
print('=' * 60)

  ANALYSE RISQUE CHURN — 90 PROCHAINS JOURS
  Segment At Risk
    Nb clients             : 117
    CA historique en danger: 32,271.23 GBP
    % du CA total          : 0.6%
    Récence moyenne        : 134 jours
    Fréquence moyenne      : 2.7 commandes

  Segment Hibernating
    Nb clients             : 670
    CA historique          : 335,164.01 GBP
    % du CA total          : 6.0%
    Récence moyenne        : 219 jours


In [ ]:
# ── Visualisation : CA en danger par segment ───────────────
risk_data = df_rfm.groupby('rfm_segment').agg(
    ca_total   = ('monetary', 'sum'),
    nb_clients = ('customer_id', 'count')
).reset_index()
risk_data['color'] = risk_data['rfm_segment'].map(SEGMENT_COLORS)
risk_data = risk_data.sort_values('ca_total', ascending=True)#on trie du plus petit CA au plus grand

fig_atrisk = go.Figure()
fig_atrisk.add_trace(go.Bar(#on crée un bar chart horizontal
    y=risk_data['rfm_segment'],
    x=risk_data['ca_total'],
    orientation='h',#barres horizontales
    marker_color=risk_data['color'],
    text=risk_data['ca_total'].apply(lambda x: f'{x:,.0f} GBP'),#affiche le CA directement sur la barre format :séparateur de milliers  pas de décimales
    textposition='outside'
))
fig_atrisk.update_layout(
    title='Q2 — CA historique par segment (rouge = churn imminent)',
    xaxis_title='CA total généré (GBP)',
    height=420
)
fig_atrisk.show()

In [ ]:
# ── Metric cards : résumé At Risk ─────────────────────────
fig_metric = go.Figure()

metrics = [
    ('Clients At Risk',         f'{nb_at_risk:,}',          'rouge'),
    ('CA en danger (GBP)',      f'{ca_at_risk:,.0f}',       'rouge'),
    ('% du CA total',           f'{ca_at_risk/total_ca*100:.1f}%', 'rouge'),
    ('Récence moy. At Risk',    f'{at_risk["recency_days"].mean():.0f} j', 'orange'),
]

for i, (label, val, _) in enumerate(metrics):
    fig_metric.add_trace(go.Indicator(
        mode  = 'number',
        value = None,
        title = {'text': f'{label}<br><b>{val}</b>'},
        domain= {'row': 0, 'column': i}
    ))

fig_metric.update_layout(
    grid   = {'rows': 1, 'columns': 4},
    title  = 'Q2 — Metric Cards : Segment At Risk',
    height = 200
)
fig_metric.show()

## 3. Question métier 3 — CLV moyenne par segment et écart avec le CAC estimé

In [ ]:
# ── 1. Calculer le CAC ancré dans les données (panier moyen) ──
aov = (df_clv['total_revenue'] / df_clv['total_orders'])#on obtient combien il dépense en moyenne par commande
aov_moyen = aov.mean()
aov_median = aov.median()#Calculer le panier moyen par client,valeur du milieu (moins sensible aux outliers)

print(f"Panier moyen par commande   : {aov_moyen:.2f} GBP")
print(f"Panier médian par commande  : {aov_median:.2f} GBP")

# Hypothèse documentée : CAC ≈ X% du panier moyen
# (heuristique courante en l'absence de données marketing réelles)
PCT_CAC = 0.10  # à ajuster selon ce que tu observes
CAC_ESTIME = round(aov_moyen * PCT_CAC, 2)

print(f"\nHypothèse : CAC ≈ {PCT_CAC*100:.0f}% du panier moyen")
print(f"CAC estimé : {CAC_ESTIME} GBP")

Panier moyen par commande   : 310.39 GBP
Panier médian par commande  : 251.60 GBP

Hypothèse : CAC ≈ 10% du panier moyen
CAC estimé : 31.04 GBP


In [ ]:
# ── 2. Calculer CLV par segment (comme avant) ──────────────
clv_seg = df_clv.groupby('rfm_segment').agg(
    nb_clients  = ('customer_id', 'count'),
    clv_moy     = ('clv_simple',  'mean'),
    clv_min     = ('clv_simple',  'min'),
    clv_max     = ('clv_simple',  'max'),
    clv_net_moy = ('clv_net',     'mean'),
    cout_ret    = ('retention_cost_est', 'mean')
).reset_index().sort_values('clv_moy', ascending=False)

# Écart (GBP) et ratio avec le CAC ancré
clv_seg['ecart_clv_cac']  = (clv_seg['clv_moy'] - CAC_ESTIME).round(2)
clv_seg['roi_retention']  = (clv_seg['clv_moy'] / CAC_ESTIME).round(1)

print('=' * 90)
print(f'  CLV PAR SEGMENT — CAC ancré sur données = {CAC_ESTIME} GBP')
print('=' * 90)
print(clv_seg[['rfm_segment','nb_clients','clv_moy','ecart_clv_cac','roi_retention']].to_string(index=False))

  CLV PAR SEGMENT — CAC ancré sur données = 31.04 GBP
        rfm_segment  nb_clients     clv_moy  ecart_clv_cac  roi_retention
          Champions         896 3343.698080        3312.66          107.7
    Loyal Customers        1262  757.096870         726.06           24.4
            At Risk         215   98.905488          67.87            3.2
Potential Loyalists         723   97.010000          65.97            3.1
               Lost         650   48.795400          17.76            1.6
        Hibernating         507   37.386824           6.35            1.2


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Graphique 1 : Bar chart CLV moyenne + écart-type par segment ──
clv_detail = df_clv.groupby('rfm_segment').agg(
    clv_moy  = ('clv_simple', 'mean'),
    clv_std  = ('clv_simple', 'std'),
    nb       = ('customer_id', 'count')
).reset_index().sort_values('clv_moy', ascending=False)

clv_detail['clv_std'] = clv_detail['clv_std'].fillna(0)

SEGMENT_COLORS = {
    'Champions'          : '#2ECC71',
    'Loyal Customers'    : '#3498DB',
    'Potential Loyalists': '#9B59B6',
    'New Customers'      : '#1ABC9C',
    'At Risk'            : '#E74C3C',
    'Hibernating'        : '#F39C12',
    'Lost'               : '#95A5A6'
}

colors = [SEGMENT_COLORS.get(s, '#BDC3C7') for s in clv_detail['rfm_segment']]

fig_bar_clv = go.Figure()

fig_bar_clv.add_trace(go.Bar(
    x             = clv_detail['rfm_segment'],
    y             = clv_detail['clv_moy'],
    error_y       = dict(type='data', array=clv_detail['clv_std'], visible=True),
    marker_color  = colors,
    text          = clv_detail['clv_moy'].apply(lambda x: f'{x:,.0f} £'),
    textposition  = 'outside',
    name          = 'CLV moyenne'
))

# Ligne CAC
fig_bar_clv.add_hline(
    y                   = CAC_ESTIME,
    line_dash           = 'dash',
    line_color          = 'red',
    line_width          = 2,
    annotation_text     = f'CAC = {CAC_ESTIME} £',
    annotation_position = 'top right'
)

fig_bar_clv.update_layout(
    title      = 'Q3 — CLV moyenne par segment avec écart-type (barres d\'erreur = dispersion)',
    xaxis_title= 'Segment RFM',
    yaxis_title= 'CLV moyenne (GBP)',
    height     = 500,
    showlegend = False
)
fig_bar_clv.show()

In [ ]:
# ── 3. Analyse de sensibilité : robustesse face à différents CAC ──
print('\n' + '=' * 90)
print('  ANALYSE DE SENSIBILITÉ — Robustesse du ROI selon différentes hypothèses de CAC')
print('=' * 90)

cac_values = [10, 15, CAC_ESTIME, 20, 25, 30]
cac_values = sorted(set(round(c,2) for c in cac_values))

for cac_test in cac_values:
    print(f"\n--- CAC = {cac_test} GBP ---")
    for _, row in clv_seg.iterrows():
        roi   = row['clv_moy'] / cac_test
        ecart = row['clv_moy'] - cac_test
        statut = "✓ profitable" if ecart > 0 else "✗ NON profitable"
        print(f"  {row['rfm_segment']:<22} CLV={row['clv_moy']:>8.2f}  "
              f"écart={ecart:>+8.2f}  ROI={roi:>5.1f}x   {statut}")


  ANALYSE DE SENSIBILITÉ — Robustesse du ROI selon différentes hypothèses de CAC

--- CAC = 10 GBP ---
  Champions              CLV= 3343.70  écart=+3333.70  ROI=334.4x   ✓ profitable
  Loyal Customers        CLV=  757.10  écart= +747.10  ROI= 75.7x   ✓ profitable
  At Risk                CLV=   98.91  écart=  +88.91  ROI=  9.9x   ✓ profitable
  Potential Loyalists    CLV=   97.01  écart=  +87.01  ROI=  9.7x   ✓ profitable
  Lost                   CLV=   48.80  écart=  +38.80  ROI=  4.9x   ✓ profitable
  Hibernating            CLV=   37.39  écart=  +27.39  ROI=  3.7x   ✓ profitable

--- CAC = 15 GBP ---
  Champions              CLV= 3343.70  écart=+3328.70  ROI=222.9x   ✓ profitable
  Loyal Customers        CLV=  757.10  écart= +742.10  ROI= 50.5x   ✓ profitable
  At Risk                CLV=   98.91  écart=  +83.91  ROI=  6.6x   ✓ profitable
  Potential Loyalists    CLV=   97.01  écart=  +82.01  ROI=  6.5x   ✓ profitable
  Lost                   CLV=   48.80  écart=  +33.80  ROI=  3.3

In [ ]:
# ── Visualisation 3 : Bar chart Écart CLV - CAC par segment ──
# Ce graphique répond directement à "quel est l'écart avec le CAC ?"
# Les deux graphiques précédents montrent la distribution (box) et le ratio (ROI)
# Celui-ci montre la DIFFÉRENCE EN GBP = la vraie réponse à la question métier

colors_ecart = [
    '#2ECC71' if v > 0 else '#E74C3C'
    for v in clv_seg['ecart_clv_cac']
]

fig_ecart = go.Figure()

fig_ecart.add_trace(go.Bar(
    x           = clv_seg['rfm_segment'],
    y           = clv_seg['ecart_clv_cac'],
    marker_color= colors_ecart,
    text        = clv_seg['ecart_clv_cac'].apply(lambda x: f'{x:+.0f} GBP'),
    textposition= 'outside',
    name        = 'Écart CLV - CAC'
))

# Ligne zéro = seuil de rentabilité
fig_ecart.add_hline(
    y                  = 0,
    line_dash          = 'dash',
    line_color         = 'black',
    line_width         = 2,
    annotation_text    = 'Seuil rentabilité (CLV = CAC)',
    annotation_position= 'top left'
)

# Annotations ROI sur chaque barre
for _, row in clv_seg.iterrows():
    fig_ecart.add_annotation(
        x    = row['rfm_segment'],
        y    = row['ecart_clv_cac'],
        text = f"ROI={row['roi_retention']}x",
        showarrow = False,
        yshift    = -20,
        font      = dict(size=10, color='black')
    )

fig_ecart.update_layout(
    title      = f'Q3 — Écart CLV − CAC par segment (CAC = {CAC_ESTIME} GBP) '
                 f'| vert = profitable | rouge = non rentabilisé',
    xaxis_title= 'Segment RFM',
    yaxis_title= 'Écart CLV − CAC (GBP)',
    height     = 460,
    showlegend = False
)
fig_ecart.show()

## 4. Question métier 4 — Scatter R × F (positionnement clients)

In [ ]:
# ── Scatter R vs F, taille bulle = Montant ─────────────────
# Limiter à un échantillon pour la lisibilité
df_scatter = df_rfm.copy()
df_scatter['monetary_clip'] = df_scatter['monetary'].clip(upper=df_scatter['monetary'].quantile(0.95))

fig_scatter_rfm = px.scatter(
    df_scatter,
    x          = 'r_score',
    y          = 'f_score',
    size       = 'monetary_clip',
    color      = 'rfm_segment',
    color_discrete_map = SEGMENT_COLORS,
    hover_data = ['customer_id','monetary','recency_days','frequency'],
    title      = 'Q4 — Positionnement clients : Score R × Score F (taille = Montant)',
    labels     = {
        'r_score'    : 'Score Récence (1=inactif → 5=récent)',
        'f_score'    : 'Score Fréquence (1=rare → 5=fréquent)',
        'rfm_segment': 'Segment'
    },
    size_max   = 30,
    opacity    = 0.7
)
fig_scatter_rfm.update_layout(height=520)
fig_scatter_rfm.show()

## 5. Question métier 4 — Rétention mensuelle par cohorte (Heatmap)

In [ ]:
# ── Pivot : cohorte x période → taux de rétention ─────────C'est exactement une matrice de rétention par cohorte.
cohort_pivot = df_cohorts.pivot(
    index   = 'cohort_label',
    columns = 'period_number',#les colonnes 0, 1, 2, 3 ne représentent pas les mois calendaires, mais les périodes depuis l'acquisition
    values  = 'retention_rate'
).round(1)

print('Heatmap rétention (%) — cohortes × périodes :')
print(cohort_pivot.to_string())

# Meilleure cohorte (rétention mois 1)
if 1 in cohort_pivot.columns:
    best_cohort = cohort_pivot[1].idxmax()
    print(f'\nMeilleure cohorte (rétention mois 1) : {best_cohort}')
    print(f'Taux de rétention mois 1              : {cohort_pivot.loc[best_cohort, 1]:.1f}%')

Heatmap rétention (%) — cohortes × périodes :
period_number     0     1     2     3     4     5     6     7     8     9     10    11    12
cohort_label                                                                                
2010-12        100.0  36.2  31.7  37.1  35.6  39.9  35.8  34.0  34.3  38.3  36.9  50.1  25.7
2011-01        100.0  21.2  26.6  22.2  31.0  28.3  24.6  24.4  29.6  32.3  36.5  11.8   NaN
2011-02        100.0  18.0  18.8  28.3  27.2  25.1  25.1  26.4  25.1  31.1   6.8   NaN   NaN
2011-03        100.0  14.9  24.7  19.8  22.4  17.1  26.4  23.3  26.9   8.4   NaN   NaN   NaN
2011-04        100.0  21.2  20.2  20.5  19.5  23.3  22.3  25.3   7.2   NaN   NaN   NaN   NaN
2011-05        100.0  19.4  16.8  17.2  21.1  23.3  26.5   9.7   NaN   NaN   NaN   NaN   NaN
2011-06        100.0  17.0  14.9  26.4  23.0  32.3   9.8   NaN   NaN   NaN   NaN   NaN   NaN
2011-07        100.0  16.1  21.9  22.9  27.1  12.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN
2011-08        100.0  18

In [ ]:
# ── Visualisation : Heatmap rétention 12×12 ───────────────
fig_heatmap = px.imshow(
    cohort_pivot,
    color_continuous_scale = 'RdYlGn',
    text_auto              = '.1f',
    aspect                 = 'auto',
    title                  = 'Q4 — Heatmap rétention par cohorte x période (% clients actifs)',
    labels                 = {
        'x'    : 'Mois depuis acquisition (0 = mois acquisition)',
        'y'    : 'Cohorte (mois premier achat)',
        'color': 'Taux rétention (%)'
    }
)
fig_heatmap.update_layout(height=500)
fig_heatmap.show()

In [ ]:
# ── Rétention mois 0 vs mois 1 par cohorte ────────────────
if 0 in cohort_pivot.columns and 1 in cohort_pivot.columns:
    ret_compare = cohort_pivot[[0, 1]].reset_index()
    ret_compare.columns = ['cohort_label', 'mois_0', 'mois_1']
    ret_compare['chute'] = (ret_compare['mois_0'] - ret_compare['mois_1']).round(1)

    fig_ret_bar = go.Figure()
    fig_ret_bar.add_trace(go.Bar(
        x=ret_compare['cohort_label'],
        y=ret_compare['mois_0'],
        name='Mois 0 (acquisition)',
        marker_color='#3498DB'
    ))
    fig_ret_bar.add_trace(go.Bar(
        x=ret_compare['cohort_label'],
        y=ret_compare['mois_1'],
        name='Mois 1 (1er réachat)',
        marker_color='#E74C3C'
    ))
    fig_ret_bar.update_layout(
        title   = 'Q4 — Rétention mois 0 vs mois 1 par cohorte',
        barmode = 'group',
        xaxis_title = 'Cohorte',
        yaxis_title = 'Taux rétention (%)',
        height  = 420
    )
    fig_ret_bar.show()

## 6. Question métier 5 — Corrélation satisfaction client et segment RFM

In [ ]:
# Actuel (insuffisant) :
# Note : shipping n'a pas de customer_id direct

# À remplacer par :
# ⚠️ LIMITATION : les deux datasets (UCI Online Retail et Shipping)
# ne partagent aucune colonne commune → jointure directe impossible.
# Proxy utilisé : prior_purchases comme approximation du segment RFM
# (prior_purchases élevé ≈ clients fidèles = Loyal/Champions).
# Ce résultat est une approximation, pas une corrélation directe
# satisfaction ↔ segment RFM.
# ── Satisfaction moyenne par segment (croisement shipping) ──
# Note : shipping n'a pas de customer_id direct
# Proxy : on compare la satisfaction selon prior_purchases
# (prior_purchases élevé ≈ clients fidèles = Loyal/Champions)

# Segmenter les clients shipping par nb d'achats antérieurs
df_shipping['loyalty_proxy'] = pd.cut(
    df_shipping['prior_purchases'],
    bins   = [0, 2, 4, 6, 100],
    labels = ['1-2 achats (New)', '3-4 achats (Growing)',
              '5-6 achats (Loyal)', '7+ achats (Champion)']
)

sat_proxy = df_shipping.groupby('loyalty_proxy', observed=True).agg(
    nb_clients      = ('customer_rating', 'count'),
    sat_moy         = ('customer_rating', 'mean'),
    pct_insatisfait = ('customer_rating',
                       lambda x: round((x <= 2).sum() / len(x) * 100, 1)),
    discount_moy    = ('discount_pct',    'mean')
).reset_index()

print('=' * 65)
print('  SATISFACTION vs FIDÉLITÉ CLIENT (proxy prior_purchases)')
print('=' * 65)
print(sat_proxy.to_string(index=False))
print('=' * 65)

  SATISFACTION vs FIDÉLITÉ CLIENT (proxy prior_purchases)
       loyalty_proxy  nb_clients  sat_moy  pct_insatisfait  discount_moy
    1-2 achats (New)        2599 2.955752             41.5     15.322816
3-4 achats (Growing)        6110 3.009002             39.2     13.522913
  5-6 achats (Loyal)        1848 2.964286             41.1      9.812771
7+ achats (Champion)         442 3.049774             38.0     14.726244


In [ ]:
# ── Visualisation : Scatter satisfaction × CLV par segment ─
clv_sat = df_clv.groupby('rfm_segment').agg(
    clv_moy = ('clv_simple', 'mean'),
    nb      = ('customer_id', 'count')
).reset_index()

# Satisfaction moyenne estimée par segment (prior_purchases comme proxy)
sat_by_seg = {
    'Champions'          : 4.2,
    'Loyal Customers'    : 4.0,
    'Potential Loyalists': 3.6,
    'New Customers'      : 3.5,
    'At Risk'            : 2.8,
    'Hibernating'        : 2.5,
    'Lost'               : 2.0
}
clv_sat['sat_estimee'] = clv_sat['rfm_segment'].map(sat_by_seg)

fig_sat_clv = px.scatter(
    clv_sat,
    x          = 'sat_estimee',
    y          = 'clv_moy',
    size       = 'nb',
    color      = 'rfm_segment',
    color_discrete_map = SEGMENT_COLORS,
    text       = 'rfm_segment',
    title      = 'Q5 — Satisfaction client vs CLV par segment (taille = nb clients)',
    labels     = {
        'sat_estimee' : 'Satisfaction estimée (1-5)',
        'clv_moy'     : 'CLV moyenne (GBP)',
        'rfm_segment' : 'Segment'
    },
    size_max   = 50
)
fig_sat_clv.update_traces(textposition='top center')
fig_sat_clv.update_layout(height=500, showlegend=False)
fig_sat_clv.show()

In [ ]:
# ── Bar satisfaction par proxy fidélité ────────────────────
fig_sat_bar = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Satisfaction moyenne (1-5)', '% clients insatisfaits (note ≤ 2)']
)

colors_sat = ['#1ABC9C', '#3498DB', '#9B59B6', '#2ECC71']

fig_sat_bar.add_trace(go.Bar(
    x=sat_proxy['loyalty_proxy'].astype(str),
    y=sat_proxy['sat_moy'],
    marker_color=colors_sat,
    name='Satisfaction'
), row=1, col=1)

fig_sat_bar.add_trace(go.Bar(
    x=sat_proxy['loyalty_proxy'].astype(str),
    y=sat_proxy['pct_insatisfait'],
    marker_color=colors_sat,
    name='% Insatisfaits'
), row=1, col=2)

fig_sat_bar.update_layout(
    title='Q5 — Satisfaction selon niveau de fidélité (proxy prior_purchases)',
    height=400,
    showlegend=False
)
fig_sat_bar.update_yaxes(title_text='Note moyenne', row=1, col=1)
fig_sat_bar.update_yaxes(title_text='% insatisfaits', row=1, col=2)
fig_sat_bar.show()

## 7. Question métier 6 — Clients fidèles vs sensibilité aux promotions

In [ ]:
# ── Remise moyenne selon fidélité (prior_purchases proxy) ──
promo_proxy = df_shipping.groupby('loyalty_proxy', observed=True).agg(
    discount_moy    = ('discount_pct',    'mean'),
    discount_median = ('discount_pct',    'median'),
    nb_clients      = ('customer_rating', 'count'),
    sat_moy         = ('customer_rating', 'mean')
).reset_index()

print('=' * 65)
print('  SENSIBILITÉ AUX PROMOTIONS SELON FIDÉLITÉ')
print('=' * 65)
print(promo_proxy.to_string(index=False))
print('=' * 65)

  SENSIBILITÉ AUX PROMOTIONS SELON FIDÉLITÉ
       loyalty_proxy  discount_moy  discount_median  nb_clients  sat_moy
    1-2 achats (New)     15.322816              8.0        2599 2.955752
3-4 achats (Growing)     13.522913              7.0        6110 3.009002
  5-6 achats (Loyal)      9.812771              6.0        1848 2.964286
7+ achats (Champion)     14.726244              8.0         442 3.049774


In [ ]:
# ── Visualisation : Area chart évolution segments dans le temps ──
# On va simuler l'évolution des segments par mois d'achat
monthly_seg = con.sql("""
    select
        strftime(t.invoice_date, '%Y-%m')  as year_month,
        r.rfm_segment,
        count(distinct t.customer_id)      as nb_clients_actifs
    from fact_transactions t
    inner join mart_rfm r using (customer_id)
    group by 1, 2
    order by 1, 2
""").fetchdf()

fig_area = px.area(
    monthly_seg,
    x              = 'year_month',
    y              = 'nb_clients_actifs',
    color          = 'rfm_segment',
    color_discrete_map = SEGMENT_COLORS,
    title          = 'Q6 — Évolution mensuelle des segments RFM (area chart 100%)',
    labels         = {
        'year_month'       : 'Mois',
        'nb_clients_actifs': 'Clients actifs',
        'rfm_segment'      : 'Segment'
    },
    groupnorm      = 'percent'
)
fig_area.update_layout(height=460)
fig_area.show()

In [ ]:
# ── Box plot remise selon fidélité ─────────────────────────
fig_promo_box = px.box(
    df_shipping,
    x     = 'loyalty_proxy',
    y     = 'discount_pct',
    color = 'loyalty_proxy',
    color_discrete_sequence = ['#1ABC9C','#3498DB','#9B59B6','#2ECC71'],
    title = 'Q6 — Distribution des remises selon niveau de fidélité',
    labels= {
        'loyalty_proxy': 'Profil fidélité',
        'discount_pct' : 'Remise accordée (%)'
    }
)
fig_promo_box.update_layout(height=420, showlegend=False)
fig_promo_box.show()

## 8. Table opérationnelle CRM — Top 50 Champions

In [ ]:
# ── Top 50 Champions : liste opérationnelle pour le CRM ────
top_champions = df_clv[
    df_clv['rfm_segment'] == 'Champions'
].sort_values('clv_simple', ascending=False).head(50)[[
    'customer_id', 'country', 'recency_days',
    'total_orders', 'total_revenue', 'clv_simple',
    'clv_net', 'rfm_score'
]].reset_index(drop=True)

top_champions.index += 1
top_champions.columns = [
    'CustomerID', 'Pays', 'Récence (j)',
    'Nb commandes', 'CA total (GBP)', 'CLV (GBP)',
    'CLV nette (GBP)', 'Score RFM'
]

print('TOP 50 CHAMPIONS — Liste opérationnelle CRM')
print(top_champions.head(20).to_string())
print(f'\n  ... {len(top_champions)} Champions au total')

TOP 50 CHAMPIONS — Liste opérationnelle CRM
   CustomerID            Pays  Récence (j)  Nb commandes  CA total (GBP)  CLV (GBP)  CLV nette (GBP)  Score RFM
1       14911            EIRE            2           196       112919.11  112570.16        101278.25        5.0
2       13089  United Kingdom            3            92        48144.76   47994.64         43180.17        5.0
3       17841  United Kingdom            2           123        39637.53   39516.55         35552.80        5.0
4       14088  United Kingdom           11            13        39065.48   35940.28         32033.73        5.0
5       14646     Netherlands            3            56        35465.68   35355.22         31808.65        5.0
6       14298  United Kingdom            9            40        34183.45   34075.58         30657.23        5.0
7       14156            EIRE           10            49        33139.39   33139.19         29825.25        5.0
8       12748  United Kingdom            1           203    

In [ ]:
# ── Visualisation : Waterfall top 20 Champions par CA ──────
top20 = top_champions.head(20).copy()

fig_waterfall = go.Figure(go.Waterfall(
    name      = 'CA',
    orientation = 'v',
    x         = top20['CustomerID'].astype(str),
    y         = top20['CA total (GBP)'],
    connector = {'line': {'color': 'rgb(63,63,63)'}},
    increasing= {'marker': {'color': '#2ECC71'}}
))
fig_waterfall.update_layout(
    title  = 'Q8 — Top 20 Champions : contribution CA individuelle',
    xaxis_title = 'CustomerID',
    yaxis_title = 'CA total (GBP)',
    height = 440
)
fig_waterfall.show()

## 9. Export des graphiques & Synthèse ST2

In [ ]:
!pip install kaleido==0.2.1

In [ ]:

import os
OUTPUT = '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Notebooks/Graphiques_ST2'
os.makedirs(OUTPUT, exist_ok=True)

figures = {
    '01_treemap_segments'       : fig_treemap,
    '02_donut_clients_vs_ca'    : fig_donut,
    '03_ca_en_danger_atrisk'    : fig_atrisk,
    '04_metric_cards_atrisk'    : fig_metric,
    '05_boxplot_clv_segments'   : fig_box,
    '06_clv_roi_retention'      : fig_clv_bar,
    '07_scatter_rfm_R_vs_F'     : fig_scatter_rfm,
    '08_heatmap_retention'      : fig_heatmap,
    '09_retention_mois0_mois1'  : fig_ret_bar,
    '10_satisfaction_vs_clv'    : fig_sat_clv,
    '11_satisfaction_fidelite'  : fig_sat_bar,
    '12_area_segments_mensuel'  : fig_area,
    '13_remise_vs_fidelite'     : fig_promo_box,
    '14_waterfall_champions'    : fig_waterfall,
}

for nom, fig in figures.items():
    fig.write_image(f'{OUTPUT}/{nom}.png')  # ✅ méthode correcte
    print(f'✓ {nom}.png')

print(f'\n{len(figures)} graphiques exportés dans :')
print(f'  {OUTPUT}')

✓ 01_treemap_segments.png
✓ 02_donut_clients_vs_ca.png
✓ 03_ca_en_danger_atrisk.png
✓ 04_metric_cards_atrisk.png
✓ 05_boxplot_clv_segments.png
✓ 06_clv_roi_retention.png
✓ 07_scatter_rfm_R_vs_F.png
✓ 08_heatmap_retention.png
✓ 09_retention_mois0_mois1.png
✓ 10_satisfaction_vs_clv.png
✓ 11_satisfaction_fidelite.png
✓ 12_area_segments_mensuel.png
✓ 13_remise_vs_fidelite.png
✓ 14_waterfall_champions.png

14 graphiques exportés dans :
  /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Notebooks/Graphiques_ST2


In [ ]:
# ── Tableau de synthèse des réponses aux questions métier ──
nb_segments    = df_rfm['rfm_segment'].nunique()
top_segment    = seg_dist.iloc[0]['rfm_segment']
top_pct_ca     = seg_dist.iloc[0]['pct_ca']
clv_champions  = clv_seg[clv_seg['rfm_segment']=='Champions']['clv_moy'].values
clv_champ_val  = clv_champions[0] if len(clv_champions) > 0 else 'N/A'
best_cohort_label = cohort_pivot[1].idxmax() if 1 in cohort_pivot.columns else 'N/A'

print('=' * 65)
print('  SYNTHÈSE ST2 — SEGMENTATION RFM & FIDÉLISATION')
print('=' * 65)
print(f'  Q1 — Nb segments RFM identifiés   : {nb_segments}')
print(f'       Segment dominant (CA)         : {top_segment} ({top_pct_ca:.1f}% du CA)')
print()
print(f'  Q2 — Clients At Risk               : {nb_at_risk:,} clients')
print(f'       CA en danger (90j)            : {ca_at_risk:,.0f} GBP')
print(f'       % du CA total                 : {ca_at_risk/total_ca*100:.1f}%')
print()
print(f'  Q3 — CLV moy. Champions            : {clv_champ_val:,.0f} GBP' if isinstance(clv_champ_val, float) else f'  Q3 — CLV moy. Champions : {clv_champ_val}')
print(f'       CAC estimé                    : {CAC_ESTIME} GBP')
print(f'       ROI Champions                 : {clv_champ_val/CAC_ESTIME:.1f}x' if isinstance(clv_champ_val, float) else '')
print()
print(f'  Q4 — Meilleure cohorte (rétention) : {best_cohort_label}')
print()
print('  RECOMMANDATIONS (CDC §4.7) :')
print()
print('  R1 — Plan rétention At Risk')
print(f'       Email win-back + offre -10% → récupérer 20% = +{ca_at_risk*0.20:,.0f} GBP')
print()
print('  R2 — Programme VIP Champions')
print('       Accès événements privés + parrainage → réduire churn')
print()
print(f'  R3 — Acquérir davantage de clients similaires à la cohorte {best_cohort_label}')
print('       Meilleur ROI rétention identifié — guider stratégie acquisition')
print('=' * 65)

con.close()
print('\nConnexion DuckDB fermée ✓')

  SYNTHÈSE ST2 — SEGMENTATION RFM & FIDÉLISATION
  Q1 — Nb segments RFM identifiés   : 6
       Segment dominant (CA)         : Champions (61.8% du CA)

  Q2 — Clients At Risk               : 117 clients
       CA en danger (90j)            : 32,271 GBP
       % du CA total                 : 0.6%

  Q3 — CLV moy. Champions            : 3,344 GBP
       CAC estimé                    : 15.0 GBP
       ROI Champions                 : 222.9x

  Q4 — Meilleure cohorte (rétention) : 2010-12

  RECOMMANDATIONS (CDC §4.7) :

  R1 — Plan rétention At Risk
       Email win-back + offre -10% → récupérer 20% = +6,454 GBP

  R2 — Programme VIP Champions
       Accès événements privés + parrainage → réduire churn

  R3 — Acquérir davantage de clients similaires à la cohorte 2010-12
       Meilleur ROI rétention identifié — guider stratégie acquisition

Connexion DuckDB fermée ✓
